In [1]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

In [2]:
df = pd.read_csv("../data/silver/my1_cleaned.csv", parse_dates=['datetime'], index_col='datetime')

# sanity checks
# checking that there's no duplicate datetime
assert df.index.is_unique
# checking that there's no NaN values in any column except for ozone and pm25
# here we temporarily allow for NaN values in ozone and pm25 because we don't want to
# penalize either dataset for missing values in the other, since they are independent measurements
# since the same covariables are present in both, we can safely drop them (done in the previous step, silver)
assert not df.drop(columns=['ozone', 'pm25']).isna().any().any()

df.head()

,ozone,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,wind_speed,wind_direction,msl_pressure,blh_m
datetime,,,,,,,,,,,
2018-07-17 13:00:00,11.27571,NaN,3449.0,0.0,1598.0,23.1,40.8,9.0,250.0,1015.8,1759.9788
2018-07-17 14:00:00,9.08044,NaN,3445.0,0.0,1403.0,23.4,42.0,8.0,270.0,1015.8,1854.9941
2018-07-17 15:00:00,10.47743,NaN,3496.0,0.0,1168.0,23.0,42.1,10.0,240.0,1015.7,1916.6765
2018-07-17 16:00:00,12.12388,NaN,3628.0,0.0,1001.0,23.0,42.7,9.0,250.0,1015.7,1846.6980
2018-07-17 17:00:00,13.32130,NaN,3675.0,0.0,577.0,22.7,45.1,8.0,250.0,1015.8,1450.4360


## Adding temporal variables

Unix time, julian day, day of week, hour of day

In [3]:
df['unix_time'] = (df.index - pd.Timestamp('1970-01-01')) // pd.Timedelta('1s')
df['julian_day'] = df.index.dayofyear
df['day_of_week'] = df.index.dayofweek
df['hour_of_day'] = df.index.hour

## Transforming variables
Transforming wind speed unit.

In [4]:
# winds speed from knots to m/s
df['wind_speed'] = df['wind_speed'] * 0.514444

df.head()

,ozone,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,wind_speed,wind_direction,msl_pressure,blh_m,unix_time,julian_day,day_of_week,hour_of_day
datetime,,,,,,,,,,,,,,,
2018-07-17 13:00:00,11.27571,NaN,3449.0,0.0,1598.0,23.1,40.8,4.629996,250.0,1015.8,1759.9788,1531832400,198,1,13
2018-07-17 14:00:00,9.08044,NaN,3445.0,0.0,1403.0,23.4,42.0,4.115552,270.0,1015.8,1854.9941,1531836000,198,1,14
2018-07-17 15:00:00,10.47743,NaN,3496.0,0.0,1168.0,23.0,42.1,5.144440,240.0,1015.7,1916.6765,1531839600,198,1,15
2018-07-17 16:00:00,12.12388,NaN,3628.0,0.0,1001.0,23.0,42.7,4.629996,250.0,1015.7,1846.6980,1531843200,198,1,16
2018-07-17 17:00:00,13.32130,NaN,3675.0,0.0,577.0,22.7,45.1,4.115552,250.0,1015.8,1450.4360,1531846800,198,1,17


# Creating hourly datasets

Hourly datasets with raw variables

In [5]:
# remove index
# drop NaN pollutants
hourly_o3 = df.dropna(subset=['ozone']).reset_index().drop(columns=['datetime', 'pm25'])
hourly_pm25 = df.dropna(subset=['pm25']).reset_index().drop(columns=['datetime', 'ozone'])

display(hourly_o3.head())
display(hourly_pm25.head())

,ozone,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,wind_speed,wind_direction,msl_pressure,blh_m,unix_time,julian_day,day_of_week,hour_of_day
0,11.27571,3449.0,0.0,1598.0,23.1,40.8,4.629996,250.0,1015.8,1759.9788,1531832400,198,1,13
1,9.08044,3445.0,0.0,1403.0,23.4,42.0,4.115552,270.0,1015.8,1854.9941,1531836000,198,1,14
2,10.47743,3496.0,0.0,1168.0,23.0,42.1,5.144440,240.0,1015.7,1916.6765,1531839600,198,1,15
3,12.12388,3628.0,0.0,1001.0,23.0,42.7,4.629996,250.0,1015.7,1846.6980,1531843200,198,1,16
4,13.32130,3675.0,0.0,577.0,22.7,45.1,4.115552,250.0,1015.8,1450.4360,1531846800,198,1,17


,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,wind_speed,wind_direction,msl_pressure,blh_m,unix_time,julian_day,day_of_week,hour_of_day
0,22.7,3609.0,0.0,1683.0,30.1,40.1,3.601108,240.0,1013.5,1739.1155,1532437200,205,1,13
1,17.8,3644.0,0.0,963.0,28.7,42.3,4.629996,240.0,1013.2,1798.7944,1532440800,205,1,14
2,19.8,3672.0,0.0,1898.0,28.8,43.6,3.086664,250.0,1013.2,1731.4312,1532444400,205,1,15
3,19.0,3507.0,0.0,1304.0,29.3,42.5,4.115552,240.0,1012.9,1655.7957,1532448000,205,1,16
4,26.7,3707.0,0.0,1199.0,29.1,40.0,4.115552,250.0,1012.9,1563.8152,1532451600,205,1,17


## Creating transformed, daily datasets

Daily means and transformed variables - easier for SR

In [6]:
# removing hour of day, and unix time (we will rebuild that as a linear time varaible)
# removing "categorical" variables (we will rebuild)
daily_df = df.drop(columns=['hour_of_day', 'unix_time', 'julian_day', 'day_of_week'])

print("Size before:", len(daily_df))

# creating means
daily_df['date'] = daily_df.index.date

daily_df = daily_df.reset_index(drop=True)

# transforming wind speed and direction into u and v components
# must be done before grouping by date
wd_rad = np.deg2rad(daily_df['wind_direction'])
daily_df['wind_u'] = -daily_df['wind_speed'] * np.sin(wd_rad) # + = east
daily_df['wind_v'] = -daily_df['wind_speed'] * np.cos(wd_rad) # + = north

daily_df = daily_df.groupby('date').mean().reset_index()

print("Size after:", len(daily_df))

# guarantee that the data is sorted by date
daily_df = daily_df.sort_values(by='date').reset_index(drop=True)

# adding a linear time variable | t = 0 for the first day, t = 1 for the last day
d = pd.to_datetime(daily_df['date'])
daily_df['t'] = (d - d.min()) / (d.max() - d.min())

# read julian day
daily_df['julian_day'] = pd.to_datetime(daily_df['date']).dt.dayofyear

# building jsin and jcos variables: 2pij/365
daily_df['jcos'] = np.cos(2 * np.pi * daily_df['julian_day'] / 365)
daily_df['jsin'] = np.sin(2 * np.pi * daily_df['julian_day'] / 365)

# jsum = jsin + jcos => highs on winter, lows on summer
daily_df['jsum'] = daily_df['jsin'] + daily_df['jcos']

# dropping aux
daily_df.drop(columns=['julian_day', 'jcos', 'jsin', 'wind_speed', 'wind_direction'], inplace=True)

# readding day of week
daily_df['day_of_week'] = pd.to_datetime(daily_df['date']).dt.dayofweek

# creating dw_wave: np.cos(1.4 * np.pi * df['dw'] / 14) + np.sin(1.2 * np.pi * df['dw'] / 7)
daily_df['dw_wave'] = np.cos(1.4 * np.pi * daily_df['day_of_week'] / 14) + np.sin(1.2 * np.pi * daily_df['day_of_week'] / 7)

# drop dw
daily_df.drop(columns=['day_of_week'], inplace=True)

daily_df.head()

Size before: 34770
Size after: 1512


,date,ozone,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,msl_pressure,blh_m,wind_u,wind_v,t,jsum,dw_wave
0,2018-07-17,11.765562,NaN,3669.090909,0.0,574.818182,21.490909,51.409091,1016.254545,1239.486965,3.677994,1.513627,0.000000,-1.228280,1.463956
1,2018-07-18,15.116736,NaN,3176.750000,0.0,750.041667,20.658333,57.316667,1018.070833,684.119487,1.660029,0.465860,0.000582,-1.240163,1.689613
2,2018-07-19,17.623141,NaN,3038.166667,0.0,798.833333,21.495833,52.841667,1017.975000,660.682934,-0.263770,0.678613,0.001163,-1.251680,1.586778
3,2018-07-20,21.776692,NaN,3155.375000,0.0,426.333333,21.020833,57.183333,1014.254167,483.383729,-0.368469,0.948179,0.001745,-1.262825,1.143590
4,2018-07-21,44.946215,NaN,3172.791667,0.0,736.375000,22.579167,60.420833,1013.520833,717.435883,1.578439,0.066848,0.002327,-1.273596,0.433884


In [7]:
# building pollutant specific dataframes
# and dropping NaN values for each pollutant
daily_o3 = daily_df.drop(columns=['date', 'pm25']).dropna(subset=['ozone'])
daily_pm25 = daily_df.drop(columns=['date', 'ozone']).dropna(subset=['pm25'])

display(daily_o3.head())
display(daily_pm25.head())

,ozone,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,msl_pressure,blh_m,wind_u,wind_v,t,jsum,dw_wave
0,11.765562,3669.090909,0.0,574.818182,21.490909,51.409091,1016.254545,1239.486965,3.677994,1.513627,0.000000,-1.228280,1.463956
1,15.116736,3176.750000,0.0,750.041667,20.658333,57.316667,1018.070833,684.119487,1.660029,0.465860,0.000582,-1.240163,1.689613
2,17.623141,3038.166667,0.0,798.833333,21.495833,52.841667,1017.975000,660.682934,-0.263770,0.678613,0.001163,-1.251680,1.586778
3,21.776692,3155.375000,0.0,426.333333,21.020833,57.183333,1014.254167,483.383729,-0.368469,0.948179,0.001745,-1.262825,1.143590
4,44.946215,3172.791667,0.0,736.375000,22.579167,60.420833,1013.520833,717.435883,1.578439,0.066848,0.002327,-1.273596,0.433884


,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,msl_pressure,blh_m,wind_u,wind_v,t,jsum,dw_wave
7,20.518182,3081.458333,0.000000,906.583333,24.066667,60.120833,1014.125000,772.554904,2.519386,1.306358,0.004072,-1.303633,1.463956
8,15.931818,3127.208333,0.000000,1011.541667,24.804167,55.800000,1015.841667,792.904168,0.539193,-0.451648,0.004654,-1.312876,1.689613
9,27.795833,3229.750000,0.000000,894.916667,26.954167,55.479167,1012.820833,775.559806,0.244523,1.086651,0.005236,-1.321731,1.586778
10,20.950000,3198.250000,0.258333,792.708333,24.316667,61.020833,1009.066667,518.339887,1.726008,0.604388,0.005817,-1.330194,1.143590
11,11.375000,3225.583333,0.008333,874.625000,19.608333,60.083333,1006.387500,1286.507911,4.016091,2.648710,0.006399,-1.338262,0.433884


In [8]:
# last check
for name, d in [('hourly_o3', hourly_o3), ('hourly_pm25', hourly_pm25),
                ('daily_o3', daily_o3), ('daily_pm25', daily_pm25)]:
    assert not d.isna().any().any(), name
    print(name, d.shape)
print(hourly_o3['unix_time'].min())   # expect ~1.53e9
print(daily_o3['t'].min(), daily_o3['t'].max())

hourly_o3 (33091, 14)
hourly_pm25 (29954, 14)
daily_o3 (1474, 13)
daily_pm25 (1349, 13)
1531832400
0.0 0.9982547993019197


# Saving data

Pollutants renamed to "target"

In [9]:
hourly_o3_out = hourly_o3.copy().rename(columns={'ozone': 'target'})
hourly_pm25_out = hourly_pm25.copy().rename(columns={'pm25': 'target'})

daily_o3_out = daily_o3.copy().rename(columns={'ozone': 'target'})
daily_pm25_out = daily_pm25.copy().rename(columns={'pm25': 'target'})

# hourly data
hourly_o3_out.to_csv("../data/gold/my1_hourly_o3.csv", index=False)
hourly_pm25_out.to_csv("../data/gold/my1_hourly_pm25.csv", index=False)

# daily data
daily_o3_out.to_csv("../data/gold/my1_daily_o3.csv", index=False)
daily_pm25_out.to_csv("../data/gold/my1_daily_pm25.csv", index=False)